# Stage 1 - Representation Learning for CIFAR10 embeddings
According to the paper, self-supervised learning methodology 'SimCLR' was used to define and train a model to produce representational embeddings of the 50k images in the CIFAR10 dataset. These embeddings are used in the TCP_RP algorithm as an input to Typical Clustering (with budget B.

# Step 1: Set up

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn.functional as F
import torch.nn as nn
from torch.utils.data import DataLoader

In [ ]:
# Access to GPU compute is advised for training -> Check if GPU is enabled
print(torch.cuda.is_available())  # should print True
print(torch.cuda.get_device_name(0))  # should print the GPU name
print(f"Memory available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Mount Google Drive so checkpoints can be saved
from google.colab import drive
drive.mount('/content/drive')

# Step 2: Prepare training data


In [ ]:
# Define augmentation pipeline according to SimCLR paper

transform = transforms.Compose([
    transforms.RandomResizedCrop(size=32, scale=(0.2, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [ ]:
# Define a wrapper class essentially so augmentation is applied twice -> results in two independent views per image.

class TwoViewDataset:
    def __init__(self, base_dataset, transform):
        self.base_dataset = base_dataset
        self.transform = transform

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, index):
        img, label = self.base_dataset[index]
        view1 = self.transform(img)
        view2 = self.transform(img)
        return (view1, view2), label

In [ ]:
# Load base CIFAR10
base_cifar10 = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=None)

# Wrap it -> create trainset and its DataLoader!
trainset = TwoViewDataset(base_dataset=base_cifar10, transform=transform)
trainloader = DataLoader(trainset, batch_size=512, shuffle=True, num_workers=2)

# Step 3: Define SimCLR model

In [ ]:
# First load the ResNet-18 classifier
resnet18 = torchvision.models.resnet18(weights=None, progress=True)

# Reduce kernel size and stride as CIFAR10 images are very small
resnet18.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)

# Remove max pooling layer for same reason
resnet18.maxpool = nn.Identity()

# Remove final classification layer (as embeddings are in penultimate layer)
resnet18.fc = nn.Identity()

In [ ]:
# Define projection head (only used during training)
class ProjectionHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(512, 512)
        self.bn = nn.BatchNorm1d(512)
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Linear(512, 128)

    def forward(self, x):
        x = self.fc1(x)
        x = self.bn(x)
        x = self.relu(x)
        x = self.fc2(x)
        return F.normalize(x, dim=1)

In [ ]:
# Define SimCLR model (a wrapper of the ResNet-18 and projection head)
class SimCLR(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = resnet18
        self.projection = ProjectionHead()

    def forward(self, x1, x2):
        h1 = self.encoder(x1)  # (batch_size, 512)
        h2 = self.encoder(x2)  # (batch_size, 512)

        z1 = self.projection(h1)  # (batch_size, 128)
        z2 = self.projection(h2)  # (batch_size, 128)

        return z1, z2

    # Use after training for embedding extraction
    def get_embedding(self, x):
        return self.encoder(x)

# Step 4: Define hyperparameters for training loop

In [ ]:
# Define loss function given in paper 
class NTXentLoss(nn.Module):
    def __init__(self, temperature=0.5):
        super().__init__()
        self.temperature = temperature

    def forward(self, z1, z2):
      out = torch.cat([z1, z2], dim=0)
      n_samples = len(out)

      # Full similarity matrix
      cov = torch.mm(out, out.t().contiguous())
      sim = torch.exp(cov / self.temperature)

      mask = ~torch.eye(n_samples, device=sim.device).bool()
      neg = sim.masked_select(mask).view(n_samples, -1).sum(dim=-1)

      # Positive similarity
      pos = torch.exp(torch.sum(z1 * z2, dim=-1) / self.temperature)
      pos = torch.cat([pos, pos], dim=0)

      loss = -torch.log(pos / neg).mean()
      return loss

In [ ]:
# Instantiate model and hyperparameterrs

model = SimCLR().to(device)
num_epochs = 500
optimiser = torch.optim.SGD(model.parameters(), lr=0.4, momentum=0.9, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=num_epochs)
criterion = NTXentLoss(temperature=0.5).to(device)

# Step 5: Train model

NB: The runtime may timeout due to activity (this model takes a few hours to train). The current best model is uploaded to the drive. If training has stopped and must be resumed, run the following cell.

In [ ]:
# Load last saved model before training paused (ONLY RUN IF YOU MUST RESUME TRAINING)

checkpoint_path = '/content/drive/MyDrive/5CCSAMLF_CW2/models/simclr_latest.pth'
checkpoint = torch.load(checkpoint_path)

model.load_state_dict(checkpoint['model_state_dict'])
optimiser.load_state_dict(checkpoint['optimiser_state_dict'])
scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
start_epoch = checkpoint['epoch'] + 1  # resume from next epoch

print(f"Resuming from epoch {start_epoch}, last loss: {checkpoint['loss']:.4f}")

In [ ]:
# Train the model

least_loss = float('inf')
checkpoint_path = '/content/drive/MyDrive/5CCSAMLF_CW2/models/simclr_latest.pth'

model.train()
for epoch in range(num_epochs): # (start_epoch, num_epochs) -> add start_epoch to range if you resume training
    total_loss = 0

    for (x1, x2), labels in trainloader:
        x1, x2 = x1.to(device), x2.to(device)

        optimiser.zero_grad()

        z1, z2 = model(x1, x2)
        loss = criterion(z1, z2)

        loss.backward()
        optimiser.step()

        total_loss += loss.item()

    scheduler.step()

    avg_loss = total_loss / len(trainloader)

    print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {avg_loss:.4f}")

    # Save the best model
    if avg_loss <= least_loss:
        least_loss = avg_loss
        torch.save({'epoch': epoch,
                  'model_state_dict': model.state_dict(),
                  'optimiser_state_dict': optimiser.state_dict(),
                  'scheduler_state_dict': scheduler.state_dict(),
                  'loss': avg_loss},
                   checkpoint_path)
        print(f"Model checkpoint at epoch {epoch+1} (best loss so far = {avg_loss})")

# Step 6: Extract train embeddings
NB: As with step 5, you may need to load the model again if the runtime disconnects -> model is no longer stored. 

In [ ]:
# Load trained model 
checkpoint_path = '/content/drive/MyDrive/5CCSAMLF_CW2/models/simclr_latest.pth'
checkpoint = torch.load(checkpoint_path)

model = SimCLR().to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval() # not training anymore

In [ ]:
# Prepare plain CIFAR-10 with NO augmentation for embedding extraction
plain_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [ ]:
# Load plain CIFAR10 trainset (50k images)
cifar10_train = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=plain_transform)
cifar10_trainloader = DataLoader(cifar10_train, batch_size=512, shuffle=False)

In [ ]:
# Extract train embeddings
train_embeddings = []
train_labels = []

with torch.no_grad():
    for imgs, labels in cifar10_trainloader:
        imgs = imgs.to(device)
        emb = model.get_embedding(imgs)         # (batch, 512)
        emb = F.normalize(emb, dim=1)           # L2 normalise
        train_embeddings.append(emb.cpu())
        train_labels.append(labels)
        print(f"Processed {len(train_embeddings) * 512} images")

train_embeddings_tensor = torch.cat(train_embeddings, dim=0)   # (50000, 512)
train_labels_tensor = torch.cat(train_labels, dim=0)           # (50000,) - ground truth, only used for evaluation

In [ ]:
# Save train embeddings
torch.save({
    'embeddings': train_embeddings_tensor,  # (50000, 512) tensor
    'labels': train_labels_tensor           # (50000,) tensor
}, '/content/drive/MyDrive/5CCSAMLF_CW2/models/train_embeddings.pth')

# Step 7: Extract test embeddings
To be used in the AL loop

In [ ]:
# Load CIFAR10 test set (10k images)
cifar10_test = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=plain_transform)
cifar10_testloader = DataLoader(cifar10_test, batch_size=512, shuffle=False)

In [ ]:
# Extract all embeddings
test_embeddings = []
test_labels = []

with torch.no_grad():
    for imgs, labels in cifar10_testloader:
        imgs = imgs.to(device)
        emb = model.get_embedding(imgs)         # (batch, 512)
        emb = F.normalize(emb, dim=1)           # L2 normalise
        test_embeddings.append(emb.cpu())
        test_labels.append(labels)
        print(f"Processed {len(test_embeddings) * 512} images")

test_embeddings_tensor = torch.cat(test_embeddings, dim=0)   # (50000, 512)
test_labels_tensor = torch.cat(test_labels, dim=0)           # (50000,) - ground truth, only used for evaluation

In [ ]:
# Save test embeddings
torch.save({
    'embeddings': test_embeddings_tensor,  # (50000, 512) tensor
    'labels': test_labels_tensor           # (50000,) tensor
}, '/content/drive/MyDrive/5CCSAMLF_CW2/models/test_embeddings.pth')